In [ ]:
from label_studio_sdk import Client
import json

# Initialize the Label Studio client
label_studio_url = 'http://35.208.227.251'
api_token = '11c87e17e3b38e044997fbb770e31515294aa328'
client = Client(url=label_studio_url, api_key=api_token)

# Get your project
project_id = 1  # Replace with your project ID
project = client.get_project(project_id)

# Get a specific task
task_id = 7  # Replace with your task ID
task = project.get_task(task_id)

# Get annotations for the task
annotations = task

# Save to JSON file
output_file = f"task_{task_id}_annotations.json"
with open(output_file, 'w') as f:
    json.dump(annotations, f, indent=4)

print(f"Saved {len(annotations)} annotations to {output_file}")

In [ ]:
import json
import cv2
import matplotlib.pyplot as plt

# Configuration
IMAGE_PATH = 'your_image.jpg'  # Path to your downloaded image
ANNOTATION_FILE = 'task_7_annotations.json'  # Your annotation file
LABEL_COLORS = {
    'male duck': 'blue',
    'female duck': 'red',
    'Juvenile duck': 'yellow',
    'Ice': 'green',
    'duck': 'orange'
}

def plot_keypoints():
    # Load image
    img = cv2.imread(IMAGE_PATH)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert to RGB for matplotlib
    
    # Load annotations
    with open(ANNOTATION_FILE) as f:
        task_data = json.load(f)
    
    # Create figure
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    
    # Plot each annotation
    for annotation in task_data.get('annotations', []):
        for result in annotation.get('result', []):
            if result['type'] == 'keypointlabels':
                value = result['value']
                
                # Get coordinates (assuming percentage coordinates)
                x = (value['x'] / 100) * img.shape[1]
                y = (value['y'] / 100) * img.shape[0]
                
                # Get label and color
                label = value['labels'][0]
                color = LABEL_COLORS.get(label, 'gray')  # Default to gray for unknown labels
                
                # Plot the point
                plt.scatter(x, y, color=color, s=100, edgecolors='white', label=label)
    
    # Create legend with unique labels
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))  # Remove duplicates
    plt.legend(by_label.values(), by_label.keys(), title='Keypoints')
    
    plt.title('Image with Keypoint Annotations')
    plt.axis('off')
    plt.show()

if __name__ == '__main__':
    plot_keypoints()

# Transfer annotation from one labelstudio instance  to another when both source and task id are known. 

In [ ]:
from label_studio_sdk import Client
import json

# SOURCE INSTANCE CONFIG
SOURCE_LS_URL = 'http://35.208.227.251'
SOURCE_API_TOKEN = '11c87e17e3b38e044997fbb770e31515294aa328'
SOURCE_PROJECT_ID = 1  # Source project ID
SOURCE_TASK_ID = 7     # Source task ID

# TARGET INSTANCE CONFIG
TARGET_LS_URL = 'http://localhost:8081/'
TARGET_API_TOKEN = ''
TARGET_PROJECT_ID = 181  # Target project ID
TARGET_TASK_ID = 70986  # Target task ID

def transfer_annotations():
    # Connect to source instance
    source_client = Client(url=SOURCE_LS_URL, api_key=SOURCE_API_TOKEN)
    source_project = source_client.get_project(SOURCE_PROJECT_ID)
    source_task = source_project.get_task(SOURCE_TASK_ID)
    
    # Get source annotations
    source_annotations = source_task.annotations
    print(f"Found {len(source_annotations)} annotations in source task")

    # Connect to target instance
    target_client = Client(url=TARGET_LS_URL, api_key=TARGET_API_TOKEN)
    target_project = target_client.get_project(TARGET_PROJECT_ID)
    
    # Verify target task exists
    try:
        target_task = target_project.get_task(TARGET_TASK_ID)
    except Exception as e:
        raise ValueError(f"Target task {TARGET_TASK_ID} not found") from e

    # Prepare annotation payload
    for annotation in source_annotations:
        # Create clean annotation payload
        payload = {
            "result": annotation.get('result', []),
            "was_cancelled": annotation.get('was_cancelled', False),
            "ground_truth": annotation.get('ground_truth', False),
            # Add other fields as needed
        }

        # Post to target instance
        response = target_client.make_request(
            'POST',
            f'/api/tasks/{TARGET_TASK_ID}/annotations/',
            json=payload
        )

        if response.status_code == 201:
            print(f"Successfully transferred annotation {annotation['id']}")
        else:
            print(f"Failed to transfer annotation {annotation['id']}: {response.text}")

    print("Annotation transfer completed")

def main():
    # Optional: Save source annotations first
    source_client = Client(url=SOURCE_LS_URL, api_key=SOURCE_API_TOKEN)
    source_task = source_client.get_project(SOURCE_PROJECT_ID).get_task(SOURCE_TASK_ID)
    
    with open(f'source_annotations_{SOURCE_TASK_ID}.json', 'w') as f:
        json.dump(source_task.annotations, f, indent=2)
    
    # Perform transfer
    transfer_annotations()

if __name__ == '__main__':
    main()

In [28]:
from label_studio_sdk import Client
import json
import os
from urllib.parse import urlparse
from pathlib import Path
from icecream import ic


# SOURCE INSTANCE CONFIG
SOURCE_LS_URL = 'http://35.208.227.251/'
SOURCE_API_TOKEN = '11c87e17e3b38e044997fbb770e31515294aa328'
SOURCE_PROJECT_ID = 1

# TARGET INSTANCE CONFIG
TARGET_LS_URL = 'http://129.97.250.147:8080/'
TARGET_API_TOKEN = 'ebdc6fa5f2c3abcd502b55d5ccc1dc0e4ae9f68d'
TARGET_PROJECT_ID = 95

def extract_filename_from_path(image_path):
    # ic(image_path)
    """Extract just the filename from the full image path"""
    if not image_path:
        return None
    
    # Handle different path formats
    if image_path.startswith('/data/upload/'):
        # ic(Path(image_path).name)
        # Extract filename from Label Studio upload path
        return Path(image_path).name
    else:
        # Handle other path formats
        return Path(image_path).name

def get_all_tasks_with_images(client, project_id, type):
    """Get all tasks from a project and create a mapping of filename to task info"""
    project = client.get_project(project_id)
    tasks = project.get_tasks()
    # ic(tasks)
    filename_to_task = {}
    
    for task in tasks:
        task_data = task.get('data', {})
        if(type=="source"):
            image_path = task_data.get('img')
        else:
            image_path = task_data.get('image')
            
        # ic(image_path)
        if image_path:
            filename = extract_filename_from_path(image_path)
            filename = filename.split('-')[-1]
            # ic(filename)
            if filename:
                filename_to_task[filename] = {
                    'task_id': task['id'],
                    'task_data': task,
                    'image_path': image_path
                }
    # ic(filename_to_task)
    return filename_to_task

def transfer_annotation_for_task(source_client, target_client, source_task_id, target_task_id, source_filename):
    """Transfer annotations from source task to target task"""
    try:
        # Get source task and annotations
        source_project = source_client.get_project(SOURCE_PROJECT_ID)
        source_task = source_project.get_task(source_task_id)
        source_annotations = source_task['annotations']

        if not source_annotations:
            print(f"  No annotations found for {source_filename}")
            return False
        
        print(f"  Found {len(source_annotations)} annotations for {source_filename}")
        
        # Verify target task exists
        target_project = target_client.get_project(TARGET_PROJECT_ID)
        try:
            target_task = target_project.get_task(target_task_id)
        except Exception as e:
            print(f"  Error: Target task {target_task_id} not found for {source_filename}")
            return False
        
        successful_transfers = 0
        
        # Transfer each annotation
        for annotation in source_annotations:
            # Create clean annotation payload
            payload = {
                "result": annotation.get('result', []),
                "was_cancelled": annotation.get('was_cancelled', False),
                "ground_truth": annotation.get('ground_truth', False),
                "created_username": annotation.get('created_username', ''),
                "updated_username": annotation.get('updated_username', ''),
                # Add other fields as needed
            }
            
            # Post to target instance
            response = target_client.make_request(
                'POST', 
                f'/api/tasks/{target_task_id}/annotations/',
                json=payload
            )
            
            if response.status_code == 201:
                successful_transfers += 1
                print(f"    ✓ Transferred annotation {annotation.get('id', 'unknown')}")
            else:
                print(f"    ✗ Failed to transfer annotation {annotation.get('id', 'unknown')}: {response.text}")
        
        print(f"  Successfully transferred {successful_transfers}/{len(source_annotations)} annotations")
        return successful_transfers > 0
        
    except Exception as e:
        print(f"  Error transferring annotations for {source_filename}: {str(e)}")
        return False

def transfer_all_matching_annotations():
    """Main function to transfer annotations for all matching image files"""
    
    print("Connecting to source instance...")
    source_client = Client(url=SOURCE_LS_URL, api_key=SOURCE_API_TOKEN)
    
    print("Connecting to target instance...")
    target_client = Client(url=TARGET_LS_URL, api_key=TARGET_API_TOKEN)
    
    print("Fetching source tasks...")
    source_tasks = get_all_tasks_with_images(source_client, SOURCE_PROJECT_ID, type = 'source')
    # ic(source_tasks)
    print(f"Found {len(source_tasks)} tasks with images in source project")
    
    print("Fetching target tasks...")
    target_tasks = get_all_tasks_with_images(target_client, TARGET_PROJECT_ID, type = 'target')
    # ic(target_tasks)
    
    print(f"Found {len(target_tasks)} tasks with images in target project")
    
    # Find matching files
    matching_files = []
    for filename in source_tasks.keys():
        if filename in target_tasks:
            matching_files.append(filename)
    
    print(f"\nFound {len(matching_files)} matching image files")
    
    if not matching_files:
        print("No matching files found. Please check that:")
        print("1. Both projects contain tasks with images")
        print("2. Image filenames match between projects")
        return
    
    # Create backup directory
    backup_dir = "annotation_backups"
    os.makedirs(backup_dir, exist_ok=True)
    
    successful_transfers = 0
    failed_transfers = 0
    
    # Process each matching file
    for i, filename in enumerate(matching_files, 1):
        print(f"\n[{i}/{len(matching_files)}] Processing: {filename}")
        
        source_task_info = source_tasks[filename]
        target_task_info = target_tasks[filename]
        
        source_task_id = source_task_info['task_id']
        target_task_id = target_task_info['task_id']
        
        print(f"  Source Task ID: {source_task_id}")
        print(f"  Target Task ID: {target_task_id}")
        
        # Create backup of source annotations
        try:
            source_project = source_client.get_project(SOURCE_PROJECT_ID)
            source_task = source_project.get_task(source_task_id)
            backup_file = os.path.join(backup_dir, f'source_annotations_{source_task_id}_{filename}.json')
            
            with open(backup_file, 'w') as f:
                json.dump(source_task['annotations'], f, indent=2)
            print(f"  Backup saved: {backup_file}")
        except Exception as e:
            print(f"  Warning: Could not create backup for {filename}: {str(e)}")
        
        # Transfer annotations
        if transfer_annotation_for_task(source_client, target_client, source_task_id, target_task_id, filename):
            successful_transfers += 1
        else:
            failed_transfers += 1
    
    # Summary
    print(f"\n{'='*50}")
    print("TRANSFER SUMMARY")
    print(f"{'='*50}")
    print(f"Total matching files: {len(matching_files)}")
    print(f"Successful transfers: {successful_transfers}")
    print(f"Failed transfers: {failed_transfers}")
    print(f"Backup location: {backup_dir}/")
    
    if failed_transfers > 0:
        print(f"\nNote: {failed_transfers} transfers failed. Check the logs above for details.")

def list_unmatched_files():
    """Helper function to list files that don't have matches"""
    print("Analyzing unmatched files...")
    
    source_client = Client(url=SOURCE_LS_URL, api_key=SOURCE_API_TOKEN)
    target_client = Client(url=TARGET_LS_URL, api_key=TARGET_API_TOKEN)
    
    source_tasks = get_all_tasks_with_images(source_client, SOURCE_PROJECT_ID)
    target_tasks = get_all_tasks_with_images(target_client, TARGET_PROJECT_ID)
    
    source_only = set(source_tasks.keys()) - set(target_tasks.keys())
    target_only = set(target_tasks.keys()) - set(source_tasks.keys())
    
    if source_only:
        print(f"\nFiles only in SOURCE ({len(source_only)}):")
        for filename in sorted(source_only):
            print(f"  - {filename}")
    
    if target_only:
        print(f"\nFiles only in TARGET ({len(target_only)}):")
        for filename in sorted(target_only):
            print(f"  - {filename}")

def main():
    """Main execution function"""
    print("Label Studio Annotation Transfer Tool")
    print("=====================================")
    
    try:
        # Uncomment the line below to see unmatched files
        # list_unmatched_files()
        
        # Perform the transfer
        transfer_all_matching_annotations()
        
    except Exception as e:
        print(f"Error: {str(e)}")
        print("Please check your configuration and try again.")

if __name__ == '__main__':
    main()

Label Studio Annotation Transfer Tool
Connecting to source instance...


Connecting to target instance...
Fetching source tasks...
Found 27 tasks with images in source project
Fetching target tasks...
Found 26 tasks with images in target project

Found 26 matching image files

[1/26] Processing: cluster_1.jpg
  Source Task ID: 1
  Target Task ID: 71025
  Backup saved: annotation_backups/source_annotations_1_cluster_1.jpg.json
  Found 1 annotations for cluster_1.jpg
    ✓ Transferred annotation 18
  Successfully transferred 1/1 annotations

[2/26] Processing: cluster_2.jpg
  Source Task ID: 2
  Target Task ID: 71026
  Backup saved: annotation_backups/source_annotations_2_cluster_2.jpg.json
  Found 1 annotations for cluster_2.jpg
    ✓ Transferred annotation 19
  Successfully transferred 1/1 annotations

[3/26] Processing: cluster_3.jpg
  Source Task ID: 3
  Target Task ID: 71031
  Backup saved: annotation_backups/source_annotations_3_cluster_3.jpg.json
  Found 1 annotations for cluster_3.jpg
    ✓ Transferred annotation 29
  Successfully transferred 1/1 anno

In [ ]:
from label_studio_sdk import Client
import json
import os
from urllib.parse import urlparse
from pathlib import Path
from icecream import ic


# SOURCE INSTANCE CONFIG
SOURCE_LS_URL = 'http://35.208.227.251/'
SOURCE_API_TOKEN = '11c87e17e3b38e044997fbb770e31515294aa328'
SOURCE_PROJECT_ID = 1

# TARGET INSTANCE CONFIG
TARGET_LS_URL = 'http://129.97.250.147:8080/'
TARGET_API_TOKEN = 'ebdc6fa5f2c3abcd502b55d5ccc1dc0e4ae9f68d'
TARGET_PROJECT_ID = 95

# LABEL MAPPING CONFIGURATION
LABEL_MAPPING = {
    "Adult Male": "male duck",
    "Adult Female": "female duck", 
    "Juvenile": "Juvenile duck",
    "Unknown": "duck",
    "Ice": "Ice"
}

def extract_filename_from_path(image_path):
    # ic(image_path)
    """Extract just the filename from the full image path"""
    if not image_path:
        return None
    
    # Handle different path formats
    if image_path.startswith('/data/upload/'):
        # ic(Path(image_path).name)
        # Extract filename from Label Studio upload path
        return Path(image_path).name
    else:
        # Handle other path formats
        return Path(image_path).name

def get_all_tasks_with_images(client, project_id, type):
    """Get all tasks from a project and create a mapping of filename to task info"""
    project = client.get_project(project_id)
    tasks = project.get_tasks()
    # ic(tasks)
    filename_to_task = {}
    
    for task in tasks:
        task_data = task.get('data', {})
        if(type=="source"):
            image_path = task_data.get('img')
        else:
            image_path = task_data.get('image')
            
        # ic(image_path)
        if image_path:
            filename = extract_filename_from_path(image_path)
            filename = filename.split('-')[-1]
            # ic(filename)
            if filename:
                filename_to_task[filename] = {
                    'task_id': task['id'],
                    'task_data': task,
                    'image_path': image_path
                }
    # ic(filename_to_task)
    return filename_to_task

def map_annotation_labels(annotation_result):
    """Map labels in annotation result from source to target format"""
    mapped_result = []
    
    for item in annotation_result:
        # Create a copy of the item to avoid modifying the original
        mapped_item = item.copy()
        
        # Check if this is a keypoint annotation with labels
        if 'value' in mapped_item and 'keypointlabels' in mapped_item['value']:
            old_labels = mapped_item['value']['keypointlabels']
            new_labels = []
            
            for old_label in old_labels:
                if old_label in LABEL_MAPPING:
                    new_labels.append(LABEL_MAPPING[old_label])
                    print(f"    Mapped label: '{old_label}' -> '{LABEL_MAPPING[old_label]}'")
                else:
                    # Keep unmapped labels as-is and warn
                    new_labels.append(old_label)
                    print(f"    Warning: No mapping found for label '{old_label}', keeping original")
            
            mapped_item['value']['keypointlabels'] = new_labels
        
        # Update the 'from_name' field if it exists and needs to be changed
        if 'from_name' in mapped_item and mapped_item['from_name'] == 'kp-1':
            mapped_item['from_name'] = 'keypoints'
            print(f"    Updated from_name: 'kp-1' -> 'keypoints'")
        
        # Update the 'to_name' field if it exists and needs to be changed  
        if 'to_name' in mapped_item and mapped_item['to_name'] == 'img-1':
            mapped_item['to_name'] = 'image'
            print(f"    Updated to_name: 'img-1' -> 'image'")
            
        mapped_result.append(mapped_item)
    
    return mapped_result

def transfer_annotation_for_task(source_client, target_client, source_task_id, target_task_id, source_filename):
    """Transfer annotations from source task to target task with label mapping"""
    try:
        # Get source task and annotations
        source_project = source_client.get_project(SOURCE_PROJECT_ID)
        source_task = source_project.get_task(source_task_id)
        source_annotations = source_task['annotations']

        if not source_annotations:
            print(f"  No annotations found for {source_filename}")
            return False
        
        print(f"  Found {len(source_annotations)} annotations for {source_filename}")
        
        # Verify target task exists
        target_project = target_client.get_project(TARGET_PROJECT_ID)
        try:
            target_task = target_project.get_task(target_task_id)
        except Exception as e:
            print(f"  Error: Target task {target_task_id} not found for {source_filename}")
            return False
        
        successful_transfers = 0
        
        # Transfer each annotation
        for annotation in source_annotations:
            print(f"    Processing annotation {annotation.get('id', 'unknown')}")
            
            # Map the annotation labels
            original_result = annotation.get('result', [])
            mapped_result = map_annotation_labels(original_result)
            
            # Create clean annotation payload with mapped labels
            payload = {
                "result": mapped_result,
                "was_cancelled": annotation.get('was_cancelled', False),
                "ground_truth": annotation.get('ground_truth', False),
                "created_username": annotation.get('created_username', ''),
                "updated_username": annotation.get('updated_username', ''),
                # Add other fields as needed
            }
            
            # Post to target instance
            response = target_client.make_request(
                'POST', 
                f'/api/tasks/{target_task_id}/annotations/',
                json=payload
            )
            
            if response.status_code == 201:
                successful_transfers += 1
                print(f"    ✓ Transferred annotation {annotation.get('id', 'unknown')} with label mapping")
            else:
                print(f"    ✗ Failed to transfer annotation {annotation.get('id', 'unknown')}: {response.text}")
        
        print(f"  Successfully transferred {successful_transfers}/{len(source_annotations)} annotations")
        return successful_transfers > 0
        
    except Exception as e:
        print(f"  Error transferring annotations for {source_filename}: {str(e)}")
        return False

def transfer_all_matching_annotations():
    """Main function to transfer annotations for all matching image files"""
    
    print("Label Mapping Configuration:")
    print("============================")
    for source_label, target_label in LABEL_MAPPING.items():
        print(f"  '{source_label}' -> '{target_label}'")
    print()
    
    print("Connecting to source instance...")
    source_client = Client(url=SOURCE_LS_URL, api_key=SOURCE_API_TOKEN)
    
    print("Connecting to target instance...")
    target_client = Client(url=TARGET_LS_URL, api_key=TARGET_API_TOKEN)
    
    print("Fetching source tasks...")
    source_tasks = get_all_tasks_with_images(source_client, SOURCE_PROJECT_ID, type = 'source')
    # ic(source_tasks)
    print(f"Found {len(source_tasks)} tasks with images in source project")
    
    print("Fetching target tasks...")
    target_tasks = get_all_tasks_with_images(target_client, TARGET_PROJECT_ID, type = 'target')
    # ic(target_tasks)
    
    print(f"Found {len(target_tasks)} tasks with images in target project")
    
    # Find matching files
    matching_files = []
    for filename in source_tasks.keys():
        if filename in target_tasks:
            matching_files.append(filename)
    
    print(f"\nFound {len(matching_files)} matching image files")
    
    if not matching_files:
        print("No matching files found. Please check that:")
        print("1. Both projects contain tasks with images")
        print("2. Image filenames match between projects")
        return
    
    # Create backup directory
    backup_dir = "annotation_backups"
    os.makedirs(backup_dir, exist_ok=True)
    
    successful_transfers = 0
    failed_transfers = 0
    
    # Process each matching file
    for i, filename in enumerate(matching_files, 1):
        print(f"\n[{i}/{len(matching_files)}] Processing: {filename}")
        
        source_task_info = source_tasks[filename]
        target_task_info = target_tasks[filename]
        
        source_task_id = source_task_info['task_id']
        target_task_id = target_task_info['task_id']
        
        print(f"  Source Task ID: {source_task_id}")
        print(f"  Target Task ID: {target_task_id}")
        
        # Create backup of source annotations
        try:
            source_project = source_client.get_project(SOURCE_PROJECT_ID)
            source_task = source_project.get_task(source_task_id)
            backup_file = os.path.join(backup_dir, f'source_annotations_{source_task_id}_{filename}.json')
            
            with open(backup_file, 'w') as f:
                json.dump(source_task['annotations'], f, indent=2)
            print(f"  Backup saved: {backup_file}")
        except Exception as e:
            print(f"  Warning: Could not create backup for {filename}: {str(e)}")
        
        # Transfer annotations
        if transfer_annotation_for_task(source_client, target_client, source_task_id, target_task_id, filename):
            successful_transfers += 1
        else:
            failed_transfers += 1
    
    # Summary
    print(f"\n{'='*50}")
    print("TRANSFER SUMMARY")
    print(f"{'='*50}")
    print(f"Total matching files: {len(matching_files)}")
    print(f"Successful transfers: {successful_transfers}")
    print(f"Failed transfers: {failed_transfers}")
    print(f"Backup location: {backup_dir}/")
    
    if failed_transfers > 0:
        print(f"\nNote: {failed_transfers} transfers failed. Check the logs above for details.")

def list_unmatched_files():
    """Helper function to list files that don't have matches"""
    print("Analyzing unmatched files...")
    
    source_client = Client(url=SOURCE_LS_URL, api_key=SOURCE_API_TOKEN)
    target_client = Client(url=TARGET_LS_URL, api_key=TARGET_API_TOKEN)
    
    source_tasks = get_all_tasks_with_images(source_client, SOURCE_PROJECT_ID, type='source')
    target_tasks = get_all_tasks_with_images(target_client, TARGET_PROJECT_ID, type='target')
    
    source_only = set(source_tasks.keys()) - set(target_tasks.keys())
    target_only = set(target_tasks.keys()) - set(source_tasks.keys())
    
    if source_only:
        print(f"\nFiles only in SOURCE ({len(source_only)}):")
        for filename in sorted(source_only):
            print(f"  - {filename}")
    
    if target_only:
        print(f"\nFiles only in TARGET ({len(target_only)}):")
        for filename in sorted(target_only):
            print(f"  - {filename}")

def validate_label_mapping():
    """Validate that all source labels have a mapping defined"""
    print("Validating label mapping...")
    
    # You can expand this to dynamically fetch labels from the projects if needed
    source_labels = {"Adult Male", "Adult Female", "Juvenile", "Unknown", "Ice"}
    target_labels = {"female duck", "male duck", "Ice", "Juvenile duck", "duck"}
    
    unmapped_source = source_labels - set(LABEL_MAPPING.keys())
    if unmapped_source:
        print(f"Warning: Source labels without mapping: {unmapped_source}")
    
    invalid_targets = set(LABEL_MAPPING.values()) - target_labels
    if invalid_targets:
        print(f"Warning: Mapped target labels not found in target schema: {invalid_targets}")
    
    print("Label mapping validation complete.\n")

def main():
    """Main execution function"""
    print("Label Studio Annotation Transfer Tool with Label Mapping")
    print("========================================================")
    
    try:
        # Validate label mapping
        validate_label_mapping()
        
        # Uncomment the line below to see unmatched files
        # list_unmatched_files()
        
        # Perform the transfer
        transfer_all_matching_annotations()
        
    except Exception as e:
        print(f"Error: {str(e)}")
        print("Please check your configuration and try again.")

if __name__ == '__main__':
    main()

In [1]:
from label_studio_sdk import Client
import json
import os
from urllib.parse import urlparse
from pathlib import Path
from icecream import ic


# SOURCE INSTANCE CONFIG
SOURCE_LS_URL = 'http://35.208.227.251/'
SOURCE_API_TOKEN = '11c87e17e3b38e044997fbb770e31515294aa328'
SOURCE_PROJECT_ID = 1

# TARGET INSTANCE CONFIG
TARGET_LS_URL = 'http://129.97.250.147:8080/'
TARGET_API_TOKEN = 'ebdc6fa5f2c3abcd502b55d5ccc1dc0e4ae9f68d'
TARGET_PROJECT_ID = 95

# LABEL MAPPING CONFIGURATION
LABEL_MAPPING = {
    "Adult Male": "male duck",
    "Adult Female": "female duck", 
    "Juvenile": "Juvenile duck",
    "Unknown": "duck",
    "Ice": "Ice"
}

def extract_filename_from_path(image_path):
    # ic(image_path)
    """Extract just the filename from the full image path"""
    if not image_path:
        return None
    
    # Handle different path formats
    if image_path.startswith('/data/upload/'):
        # ic(Path(image_path).name)
        # Extract filename from Label Studio upload path
        return Path(image_path).name
    else:
        # Handle other path formats
        return Path(image_path).name

def get_all_tasks_with_images(client, project_id, type):
    """Get all tasks from a project and create a mapping of filename to task info"""
    project = client.get_project(project_id)
    tasks = project.get_tasks()
    # ic(tasks)
    filename_to_task = {}
    
    for task in tasks:
        task_data = task.get('data', {})
        if(type=="source"):
            image_path = task_data.get('img')
        else:
            image_path = task_data.get('image')
            
        # ic(image_path)
        if image_path:
            filename = extract_filename_from_path(image_path)
            filename = filename.split('-')[-1]
            # ic(filename)
            if filename:
                filename_to_task[filename] = {
                    'task_id': task['id'],
                    'task_data': task,
                    'image_path': image_path
                }
    # ic(filename_to_task)
    return filename_to_task

def map_annotation_labels(annotation_result):
    """Map labels in annotation result from source to target format"""
    mapped_result = []
    
    for item in annotation_result:
        # Create a copy of the item to avoid modifying the original
        mapped_item = item.copy()
        
        # Check if this is a keypoint annotation with labels
        if 'value' in mapped_item and 'keypointlabels' in mapped_item['value']:
            old_labels = mapped_item['value']['keypointlabels']
            new_labels = []
            
            for old_label in old_labels:
                if old_label in LABEL_MAPPING:
                    new_labels.append(LABEL_MAPPING[old_label])
                    print(f"    Mapped label: '{old_label}' -> '{LABEL_MAPPING[old_label]}'")
                else:
                    # Keep unmapped labels as-is and warn
                    new_labels.append(old_label)
                    print(f"    Warning: No mapping found for label '{old_label}', keeping original")
            
            mapped_item['value']['keypointlabels'] = new_labels
        
        # Update the 'from_name' field if it exists and needs to be changed
        if 'from_name' in mapped_item and mapped_item['from_name'] == 'kp-1':
            mapped_item['from_name'] = 'keypoints'
            print(f"    Updated from_name: 'kp-1' -> 'keypoints'")
        
        # Update the 'to_name' field if it exists and needs to be changed  
        if 'to_name' in mapped_item and mapped_item['to_name'] == 'img-1':
            mapped_item['to_name'] = 'image'
            print(f"    Updated to_name: 'img-1' -> 'image'")
            
        mapped_result.append(mapped_item)
    
    return mapped_result

def transfer_annotation_for_task(source_client, target_client, source_task_id, target_task_id, source_filename):
    """Transfer annotations from source task to target task with label mapping"""
    try:
        # Get source task and annotations
        source_project = source_client.get_project(SOURCE_PROJECT_ID)
        source_task = source_project.get_task(source_task_id)
        source_annotations = source_task['annotations']

        if not source_annotations:
            print(f"  No annotations found for {source_filename}")
            return False
        
        print(f"  Found {len(source_annotations)} annotations for {source_filename}")
        
        # Verify target task exists
        target_project = target_client.get_project(TARGET_PROJECT_ID)
        try:
            target_task = target_project.get_task(target_task_id)
        except Exception as e:
            print(f"  Error: Target task {target_task_id} not found for {source_filename}")
            return False
        
        successful_transfers = 0
        
        # Transfer each annotation
        for annotation in source_annotations:
            print(f"    Processing annotation {annotation.get('id', 'unknown')}")
            
            # Map the annotation labels
            original_result = annotation.get('result', [])
            mapped_result = map_annotation_labels(original_result)
            
            # Create clean annotation payload with mapped labels
            payload = {
                "result": mapped_result,
                "was_cancelled": annotation.get('was_cancelled', False),
                "ground_truth": annotation.get('ground_truth', False),
                "created_username": annotation.get('created_username', ''),
                "updated_username": annotation.get('updated_username', ''),
                # Add other fields as needed
            }
            
            # Post to target instance
            response = target_client.make_request(
                'POST', 
                f'/api/tasks/{target_task_id}/annotations/',
                json=payload
            )
            
            if response.status_code == 201:
                successful_transfers += 1
                print(f"    ✓ Transferred annotation {annotation.get('id', 'unknown')} with label mapping")
            else:
                print(f"    ✗ Failed to transfer annotation {annotation.get('id', 'unknown')}: {response.text}")
        
        print(f"  Successfully transferred {successful_transfers}/{len(source_annotations)} annotations")
        return successful_transfers > 0
        
    except Exception as e:
        print(f"  Error transferring annotations for {source_filename}: {str(e)}")
        return False

def transfer_all_matching_annotations():
    """Main function to transfer annotations for all matching image files"""
    
    print("Label Mapping Configuration:")
    print("============================")
    for source_label, target_label in LABEL_MAPPING.items():
        print(f"  '{source_label}' -> '{target_label}'")
    print()
    
    print("Connecting to source instance...")
    source_client = Client(url=SOURCE_LS_URL, api_key=SOURCE_API_TOKEN)
    
    print("Connecting to target instance...")
    target_client = Client(url=TARGET_LS_URL, api_key=TARGET_API_TOKEN)
    
    print("Fetching source tasks...")
    source_tasks = get_all_tasks_with_images(source_client, SOURCE_PROJECT_ID, type = 'source')
    # ic(source_tasks)
    print(f"Found {len(source_tasks)} tasks with images in source project")
    
    print("Fetching target tasks...")
    target_tasks = get_all_tasks_with_images(target_client, TARGET_PROJECT_ID, type = 'target')
    # ic(target_tasks)
    
    print(f"Found {len(target_tasks)} tasks with images in target project")
    
    # Find matching files
    matching_files = []
    for filename in source_tasks.keys():
        if filename in target_tasks:
            matching_files.append(filename)
    
    print(f"\nFound {len(matching_files)} matching image files")
    
    if not matching_files:
        print("No matching files found. Please check that:")
        print("1. Both projects contain tasks with images")
        print("2. Image filenames match between projects")
        return
    
    # Create backup directory
    backup_dir = "annotation_backups"
    os.makedirs(backup_dir, exist_ok=True)
    
    successful_transfers = 0
    failed_transfers = 0
    
    # Process each matching file
    for i, filename in enumerate(matching_files, 1):
        print(f"\n[{i}/{len(matching_files)}] Processing: {filename}")
        
        source_task_info = source_tasks[filename]
        target_task_info = target_tasks[filename]
        
        source_task_id = source_task_info['task_id']
        target_task_id = target_task_info['task_id']
        
        print(f"  Source Task ID: {source_task_id}")
        print(f"  Target Task ID: {target_task_id}")
        
        # Create backup of source annotations
        try:
            source_project = source_client.get_project(SOURCE_PROJECT_ID)
            source_task = source_project.get_task(source_task_id)
            backup_file = os.path.join(backup_dir, f'source_annotations_{source_task_id}_{filename}.json')
            
            with open(backup_file, 'w') as f:
                json.dump(source_task['annotations'], f, indent=2)
            print(f"  Backup saved: {backup_file}")
        except Exception as e:
            print(f"  Warning: Could not create backup for {filename}: {str(e)}")
        
        # Transfer annotations
        if transfer_annotation_for_task(source_client, target_client, source_task_id, target_task_id, filename):
            successful_transfers += 1
        else:
            failed_transfers += 1
    
    # Summary
    print(f"\n{'='*50}")
    print("TRANSFER SUMMARY")
    print(f"{'='*50}")
    print(f"Total matching files: {len(matching_files)}")
    print(f"Successful transfers: {successful_transfers}")
    print(f"Failed transfers: {failed_transfers}")
    print(f"Backup location: {backup_dir}/")
    
    if failed_transfers > 0:
        print(f"\nNote: {failed_transfers} transfers failed. Check the logs above for details.")

def list_unmatched_files():
    """Helper function to list files that don't have matches"""
    print("Analyzing unmatched files...")
    
    source_client = Client(url=SOURCE_LS_URL, api_key=SOURCE_API_TOKEN)
    target_client = Client(url=TARGET_LS_URL, api_key=TARGET_API_TOKEN)
    
    source_tasks = get_all_tasks_with_images(source_client, SOURCE_PROJECT_ID, type='source')
    target_tasks = get_all_tasks_with_images(target_client, TARGET_PROJECT_ID, type='target')
    
    source_only = set(source_tasks.keys()) - set(target_tasks.keys())
    target_only = set(target_tasks.keys()) - set(source_tasks.keys())
    
    if source_only:
        print(f"\nFiles only in SOURCE ({len(source_only)}):")
        for filename in sorted(source_only):
            print(f"  - {filename}")
    
    if target_only:
        print(f"\nFiles only in TARGET ({len(target_only)}):")
        for filename in sorted(target_only):
            print(f"  - {filename}")

def validate_label_mapping():
    """Validate that all source labels have a mapping defined"""
    print("Validating label mapping...")
    
    # You can expand this to dynamically fetch labels from the projects if needed
    source_labels = {"Adult Male", "Adult Female", "Juvenile", "Unknown", "Ice"}
    target_labels = {"female duck", "male duck", "Ice", "Juvenile duck", "duck"}
    
    unmapped_source = source_labels - set(LABEL_MAPPING.keys())
    if unmapped_source:
        print(f"Warning: Source labels without mapping: {unmapped_source}")
    
    invalid_targets = set(LABEL_MAPPING.values()) - target_labels
    if invalid_targets:
        print(f"Warning: Mapped target labels not found in target schema: {invalid_targets}")
    
    print("Label mapping validation complete.\n")

def main():
    """Main execution function"""
    print("Label Studio Annotation Transfer Tool with Label Mapping")
    print("========================================================")
    
    try:
        # Validate label mapping
        validate_label_mapping()
        
        # Uncomment the line below to see unmatched files
        # list_unmatched_files()
        
        # Perform the transfer
        transfer_all_matching_annotations()
        
    except Exception as e:
        print(f"Error: {str(e)}")
        print("Please check your configuration and try again.")

if __name__ == '__main__':
    main()

Label Studio Annotation Transfer Tool with Label Mapping
Validating label mapping...
Label mapping validation complete.

Label Mapping Configuration:
  'Adult Male' -> 'male duck'
  'Adult Female' -> 'female duck'
  'Juvenile' -> 'Juvenile duck'
  'Unknown' -> 'duck'
  'Ice' -> 'Ice'

Connecting to source instance...
Connecting to target instance...
Fetching source tasks...
Found 27 tasks with images in source project
Fetching target tasks...
Found 26 tasks with images in target project

Found 26 matching image files

[1/26] Processing: cluster_1.jpg
  Source Task ID: 1
  Target Task ID: 71025
  Backup saved: annotation_backups/source_annotations_1_cluster_1.jpg.json
  Found 1 annotations for cluster_1.jpg
    Processing annotation 18
    Mapped label: 'Adult Male' -> 'male duck'
    Updated from_name: 'kp-1' -> 'keypoints'
    Updated to_name: 'img-1' -> 'image'
    Mapped label: 'Adult Male' -> 'male duck'
    Updated from_name: 'kp-1' -> 'keypoints'
    Updated to_name: 'img-1' -> '

In [ ]:
import json
import cv2
import numpy as np


from label_studio_sdk import Client
import json

# Initialize the Label Studio client
label_studio_url = 'http://35.208.227.251'
api_token = '11c87e17e3b38e044997fbb770e31515294aa328'
client = Client(url=label_studio_url, api_key=api_token)

# Get your project
project_id = 1  # Replace with your project ID
project = client.get_project(project_id)

# Get a specific task
task_id = 31  # Replace with your task ID
task = project.get_task(task_id)

# Get annotations for the task
annotations = task

# Save to JSON file
output_file = f"task_{task_id}_annotations.json"
with open(output_file, 'w') as f:
    json.dump(annotations, f, indent=4)

print(f"Saved {len(annotations)} annotations to {output_file}")

# Configuration
IMAGE_PATH = '/Users/aqeelpatel/Documents/Personal website/muhammed10patel.github.io/assets/img/project_6/aed014f4-IMG_4641.JPG'          # Path to your downloaded image
ANNOTATION_FILE = output_file  # Your annotation file
OUTPUT_IMAGE = '/Users/aqeelpatel/Documents/Personal website/muhammed10patel.github.io/assets/img/project_6/aed014f4-IMG_4641_annotated.png'   # Output file name

# Label colors in BGR format (OpenCV uses BGR instead of RGB)
LABEL_COLORS = {
    'Adult Male': (255, 0, 0),         # Blue
    'Adult Female': (0, 0, 255),       # Red
    'Juvenile': (0, 255, 255),   # Yellow
    'Ice': (0, 255, 0),               # Green
    'Unknown': (0, 165, 255)             # Orange
}

# Keypoint display settings
POINT_RADIUS = 4
BORDER_THICKNESS = 1
TEXT_THICKNESS = 2
TEXT_SCALE = 1.2

def plot_and_save_keypoints():
    # Load image
    img = cv2.imread(IMAGE_PATH)
    if img is None:
        print(f"Error: Could not load image from {IMAGE_PATH}")
        return

    # Load annotations
    with open(ANNOTATION_FILE) as f:
        task_data = json.load(f)

    # Draw annotations
    for annotation in task_data.get('annotations', []):
        for result in annotation.get('result', []):
            if result['type'] == 'keypointlabels':
                value = result['value']
                
                # Convert percentage coordinates to absolute pixels
                x = int((value['x'] / 100) * img.shape[1])
                y = int((value['y'] / 100) * img.shape[0])
                
                # Get label and color
                label = value['keypointlabels'][0]
                color = LABEL_COLORS.get(label, (128, 128, 128))  # Default to gray

                # Draw keypoint with border
                cv2.circle(img, (x, y), POINT_RADIUS + BORDER_THICKNESS, (255, 255, 255), -1)
                cv2.circle(img, (x, y), POINT_RADIUS, color, -1)
                
                # # Add label text
                # cv2.putText(img, label, (x + POINT_RADIUS + 5, y + 5),
                #             cv2.FONT_HERSHEY_SIMPLEX, TEXT_SCALE,
                #             (255, 255, 255), TEXT_THICKNESS)

    # Add legend
    legend_x = 50
    legend_y = 50
    for i, (label, color) in enumerate(LABEL_COLORS.items()):
        y_pos = legend_y + i * 50
        # Draw color swatch
        cv2.rectangle(img, (legend_x, y_pos), (legend_x + 30, y_pos + 30), color, -1)
        # Add label text
        cv2.putText(img, label, (legend_x + 40, y_pos + 25), 
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    # Save high-quality image
    cv2.imwrite(OUTPUT_IMAGE, img, [cv2.IMWRITE_JPEG_QUALITY, 100])
    print(f"Saved annotated image to {OUTPUT_IMAGE} with high resolution")

    # Optional: Display the image (press any key to close)
    # cv2.imshow('Annotated Image', img)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()

if __name__ == '__main__':
    plot_and_save_keypoints()